In [ ]:
from pathlib import Path
from typing import List, Dict, Tuple, Union
from collections import defaultdict
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm.auto import tqdm
from PIL import Image
import h5py
import scipy.io

import nilearn as nl
import nilearn.image as nl_image
import nilearn.plotting as nl_plotting
import nibabel as nib

In [ ]:
SUBJECTS = [
    f"subj{i:02d}" for i in range(1, 9)
]
data_space = "func1pt8mm"
beta_type ="betas_fithrf_GLMdenoise_RR"

# SUBJECTS

In [ ]:
ROIS = {
    
}

In [ ]:
def load_ctab(file_path):
    ctab = pd.read_csv(
        file_path,
        sep="\s+",   # split on any whitespace
        header=None,
        comment='#',             # just in case there are comments
        # skiprows=1,               # skip the "num entries" line,
        names=['roi_id', 'roi_name']
    )
    return ctab

In [ ]:
ds_path = '${MBS_NSD_DIR}'
# ds_path = '${MBS_NSD_DIR}/nsddata_betas/ppdata/subj01/fsaverage/betas_fithrf_GLMdenoise_RR'

ds_path = Path(ds_path)

# list((ds_path ).iterdir())

In [ ]:
ncsnr2nc = lambda x: 100 * (x**2) / (x**2 + 1/3)

In [ ]:
noiose_ceiling_masks = {}

for sub in tqdm(SUBJECTS):
    nc_filepath = ds_path / "nsddata_betas/ppdata" / sub / data_space / beta_type / "ncsnr.nii.gz"

    nc = nib.load(nc_filepath).get_fdata()
    nc = np.squeeze(nc)

    noiose_ceiling_masks[sub] = ncsnr2nc(nc)

    

In [ ]:
thresh = 10

for sub, mask in noiose_ceiling_masks.items():
    print(f"Subject: {sub}, available voxels: {(mask > thresh).sum()}")



In [ ]:
roi_files = [
    "streams", 
    "prf-visualrois",
    "nsdgeneral",
    "floc-words",
    "floc-places",
    "floc-faces",
    "floc-bodies"
]

In [ ]:
sub = "subj01"

roi1 = "streams"
roi_file1 = ds_path / "nsddata/freesurfer" / sub / "label" / f"lh.{roi1}.mgz"
roi_metadata1 = ds_path / "nsddata/freesurfer" / sub / "label" / f"{roi1}.mgz.ctab"

roi1 = nib.load(roi_file1).get_fdata()
roi_metadata1 = load_ctab(roi_metadata1)
roi_metadata1

In [ ]:
roi1.shape

In [ ]:
sub = "subj01"

roi2 = "prf-visualrois"
roi_file2 = ds_path / "nsddata/freesurfer" / sub / "label" / f"lh.{roi2}.mgz"
roi_metadata2 = ds_path / "nsddata/freesurfer" / sub / "label" / f"{roi2}.mgz.ctab"

roi2 = nib.load(roi_file2).get_fdata()
roi_metadata2 = load_ctab(roi_metadata2)
roi_metadata2

In [ ]:
sub = "subj01"

roi3 = "nsdgeneral"
roi_file3 = ds_path / "nsddata/freesurfer" / sub / "label" / f"lh.{roi3}.mgz"
roi_metadata3 = ds_path / "nsddata/freesurfer" / sub / "label" / f"{roi3}.mgz.ctab"

roi3 = nib.load(roi_file3).get_fdata()
roi_metadata3 = load_ctab(roi_metadata3)
roi_metadata3

In [ ]:
sub = "subj01"

roi4 = "floc-words"
roi_file4 = ds_path / "nsddata/freesurfer" / sub / "label" / f"lh.{roi4}.mgz"
roi_metadata4 = ds_path / "nsddata/freesurfer" / sub / "label" / f"{roi4}.mgz.ctab"

roi4 = nib.load(roi_file4).get_fdata()
roi_metadata4 = load_ctab(roi_metadata4)
roi_metadata4

In [ ]:
# Iterate combinations of ROIs
from itertools import product

roi_x = roi1
roi_y = roi2
roi_m1 = roi_metadata1
roi_m2 = roi_metadata2

roi_x = roi1
roi_y = roi3
roi_m1 = roi_metadata1
roi_m2 = roi_metadata3

roi_x = roi2
roi_y = roi3
roi_m1 = roi_metadata2
roi_m2 = roi_metadata3

roi_x = roi4
roi_y = roi3
roi_m1 = roi_metadata4
roi_m2 = roi_metadata3

roi_x = roi4
roi_y = roi2
roi_m1 = roi_metadata4
roi_m2 = roi_metadata2

roi_x = roi4
roi_y = roi1
roi_m1 = roi_metadata4
roi_m2 = roi_metadata1

roi_combinations = product(roi_m1.roi_name[1:].tolist(), roi_m2.roi_name[1:].tolist())
roi_combinations = product(roi_m1.roi_name[1:].tolist(), roi_m2.roi_name[1:].tolist())

for roi_1, roi_2 in roi_combinations:
    id1 = roi_m1[roi_m1.roi_name == roi_1].roi_id.values[0]
    mask1 = roi_x == id1
    
    id2 = roi_m2[roi_m2.roi_name == roi_2].roi_id.values[0]
    mask2 = roi_y == id2
    
    intersection = np.logical_and(mask1, mask2)

    print(f"{roi_1}: {mask1.sum()}, {roi_2}: {mask2.sum()}, intersection: {intersection.sum()}")